# TauswortheGen — taus88 equidistribution

**Tausworthe** generators (L'Ecuyer 1996) are LFSRs over $\mathbb{F}_2$
defined by a primitive polynomial and a decimation step $s$. Single
components have modest equidistribution; the textbook combined
construction taus88 XORs three Tausworthe streams to bring the
combined period to $\approx 2^{88}$ and the equidistribution gap
profile to a small constant.

This notebook builds the *first* component of taus88 ($z^{31} +
z^{13} + 1$, $s = 12$). See
[generators/TauswortheGen.md](../../generators/TauswortheGen.md) for
the full three-component recipe.


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _L'Ecuyer (1996)_


In [ ]:
# taus88 component 1 (L'Ecuyer 1996 — Table III). The full taus88
# combines this with two more Tausworthe streams (k=29, k=28).
gen = Generator.create("TauswortheGen", L=31,
    k=31, nb_terms=3, quicktaus=True,
    poly=[0, 13, 31], s=12)
print(gen.display())


## Wrap in a `Combination`


In [ ]:
# Wrap the generator in a single-component Combination. The
# Combination is the live object the search loop iterates and the
# equidistribution test consumes.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

Tausworthe generators are full-period when the polynomial is primitive; **Harase** applies.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "taus88"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('taus88')
# entry.components[0] carries the same params as constructed above
```
